# Data Cleaning 03 -- AAII Sentiment

## Input
`Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_post` (Parquet, 1,095 rows x 13 columns)

## Purpose
Cleans the weekly AAII (American Association of Individual Investors) sentiment survey. Keyed on `date` only (no PERMNO). Key concerns addressed: verifying that sentiment percentages sum to ~100%, identifying redundant columns (AAII historical summaries, S&P 500 weekly prices already available from other sources), checking units (decimal vs percentage), and handling a junk overflow column from header parsing.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification.

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with dtype information
- Per-row NaN distribution and sample rows with missing values

## Stage 2: Sentiment-Specific Checks
- **Day-of-week distribution:** confirms survey dates are typically Thursdays
- **Date frequency:** reports gap statistics (mean, median, min, max) and gap distribution; expected ~7 days between observations
- **Sentiment sum check:** verifies `bullish + neutral + bearish` sums to ~100 (or ~1.0 in decimal) for every row, flags any rows outside tolerance
- **Value range checks:** validates sentiment columns fall within expected bounds (0--100 for percentages or 0--1 for decimals; -60 to +60 for bull_bear_spread) and reports S&P 500 column ranges
- **Units check:** determines whether values are in decimal (0--1) or percentage (0--100) by examining means
- **Redundancy check:** identifies AAII backward-looking summary columns (`bullish_avg`, `bullish_avg_plus_sd`, `bullish_avg_minus_sd`, `total`) and S&P 500 columns that duplicate data from other pipeline sources
- **Duplicate date check**

## Stage 4: Clean & Save

### Columns Dropped (8)
- `extra_0` -- junk overflow column from header parsing, 100% None (object dtype)
- `total` -- always equals `bullish + neutral + bearish` = 1.0000, carries zero information
- `bullish_avg`, `bullish_avg_plus_sd`, `bullish_avg_minus_sd` -- constants. These are AAII's fixed historical averages printed identically on every row of their spreadsheet (0.376304, 0.476562, 0.276046 respectively). They do not vary over time and have no predictive value.
- `sp500_weekly_high`, `sp500_weekly_low`, `sp500_weekly_close` -- redundant with S&P 500 data already in the pipeline from CRSP daily returns and Fama-French market factor

### No NaN Handling Needed
All 5 retained columns have zero NaN across all 1,095 rows.

### Units Left as Decimal (0--1)
The collection code converted "62.41%" to 0.6241. This is internally consistent -- `bullish + neutral + bearish = 1.0` for every row. No conversion to percentage applied since the model normalises all inputs.

### Date Range
Already 2004-01-01 to 2024-12-26 from collection. No trim needed.

### Publication Lag Not Enforced Here
AAII releases on Thursday. The merge pipeline must handle this when forward-filling to daily.

### Factors Retained (5)
- `bullish` -- % bullish respondents (decimal)
- `neutral` -- % neutral respondents (decimal)
- `bearish` -- % bearish respondents (decimal)
- `bullish_8w_ma` -- AAII's 8-week moving average of bullish % (smoothed signal)
- `bull_bear_spread` -- bullish minus bearish (headline indicator)

## Output
`Data/Data_Collection/Cleaned/03_AAII_Sentiment/aaii_sentiment_clean.parquet` -- 5 factor columns (down from 13 before cleaning)

In [1]:
# %% [markdown]
# # Data Cleaning: AAII_sentiment_post
#
# Source: Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_post
# Output: Data/Data_Collection/Cleaned/03_AAII_Sentiment/aaii_sentiment_clean.parquet
#
# Weekly AAII (American Association of Individual Investors) sentiment survey.
# No PERMNO — keyed on date only (survey date, typically Thursday).
#
# Key concerns:
#   - Sentiment columns (bullish/neutral/bearish) should sum to ~100%
#   - The historical averages and +/-SD columns are backward-looking summaries
#     from AAII, not factors — may want to drop and compute our own
#   - S&P 500 weekly OHLC columns are redundant with CRSP/FRED data
#   - No publication lag column — AAII releases on Thursday; downstream
#     pipeline must handle this
#   - Values should be in percentage terms (0-100)

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/03_AAII_Sentiment/AAII_sentiment_post')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/03_AAII_Sentiment')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — AAII Sentiment")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique():,}")

factor_cols = [c for c in df.columns if c != 'date']

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<30s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT — AAII Sentiment")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<30s} {'NaN %':>8s}  {'Count':>6s}  {'dtype':>10s}")
print("  " + "-" * 60)
for col in factor_cols:
    n = df[col].isna().sum()
    pct = n / n_rows * 100
    status = "✓" if n == 0 else "⚠"
    print(f"  {status} {col:<28s} {pct:>7.2f}%  {n:>6d}  {str(df[col].dtype):>10s}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>6,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>6,d}")
print(f"  Rows with >3 NaN: {(row_nan > 3).sum():>6,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

if (row_nan > 0).any():
    print(f"\n  Sample rows with NaN:")
    print(df[row_nan > 0].head(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: SENTIMENT-SPECIFIC CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: SENTIMENT-SPECIFIC CHECKS")
print("=" * 90)

# ── 2a. Day-of-week distribution ────────────────────────────────────────────
print(f"\n--- Day-of-week distribution ---")
dow_counts = df['date'].dt.day_name().value_counts()
print(dow_counts.to_string())

# ── 2b. Date frequency ──────────────────────────────────────────────────────
print(f"\n--- Date gap distribution ---")
date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Mean gap: {date_diffs.mean():.1f} days")
print(f"  Median gap: {date_diffs.median():.0f} days")
print(f"  Min gap: {date_diffs.min():.0f} days")
print(f"  Max gap: {date_diffs.max():.0f} days")
print(f"\n  Gap distribution:")
for gap, count in date_diffs.value_counts().sort_index().head(10).items():
    print(f"    {int(gap):>3d} days: {count:>5,d}")

# ── 2c. Sentiment sum check ─────────────────────────────────────────────────
# bullish + neutral + bearish should sum to ~100%
print(f"\n--- Sentiment sum check (bullish + neutral + bearish) ---")
if all(c in df.columns for c in ['bullish', 'neutral', 'bearish']):
    sentiment_sum = df['bullish'] + df['neutral'] + df['bearish']
    print(f"  Mean sum:   {sentiment_sum.mean():.2f} (expect ~100)")
    print(f"  Min sum:    {sentiment_sum.min():.2f}")
    print(f"  Max sum:    {sentiment_sum.max():.2f}")
    print(f"  Std:        {sentiment_sum.std():.4f}")

    n_bad = ((sentiment_sum < 99) | (sentiment_sum > 101)).sum()
    if n_bad > 0:
        print(f"  ⚠ {n_bad} rows where sum is outside [99, 101]")
    else:
        print(f"  ✓ All rows sum to ~100")
else:
    print(f"  ⚠ Missing one or more of bullish/neutral/bearish columns")

# ── 2d. Value range checks ──────────────────────────────────────────────────
print(f"\n--- Value range checks ---")

# Sentiment columns should be 0-100 (percentages)
sentiment_cols = ['bullish', 'neutral', 'bearish', 'bullish_8w_ma',
                  'bull_bear_spread', 'bullish_avg', 'bullish_avg_plus_sd',
                  'bullish_avg_minus_sd']

for col in sentiment_cols:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue

    if col == 'bull_bear_spread':
        # Spread can be negative
        expected_low, expected_high = -60, 60
    else:
        expected_low, expected_high = 0, 100

    n_out = ((vals < expected_low) | (vals > expected_high)).sum()
    if n_out > 0:
        print(f"  ⚠ {col:<30s} {n_out:>3d} values outside [{expected_low}, {expected_high}]")
        print(f"    Actual range: {vals.min():.2f} to {vals.max():.2f}")
    else:
        print(f"  ✓ {col:<30s} range: {vals.min():.2f} to {vals.max():.2f}")

# S&P 500 columns
sp_cols = ['sp500_weekly_high', 'sp500_weekly_low', 'sp500_weekly_close']
for col in sp_cols:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    if len(vals) > 0:
        print(f"  ✓ {col:<30s} range: {vals.min():.0f} to {vals.max():.0f}")

# ── 2e. Check for units issue ───────────────────────────────────────────────
# Are values 0-1 (decimal) or 0-100 (percentage)?
print(f"\n--- Units check (decimal vs percentage) ---")
for col in ['bullish', 'neutral', 'bearish']:
    if col not in df.columns:
        continue
    mean_val = df[col].dropna().mean()
    if mean_val < 1:
        print(f"  ⚠ {col}: mean = {mean_val:.4f} — appears to be DECIMAL (0-1)")
    elif mean_val > 1 and mean_val < 100:
        print(f"  ✓ {col}: mean = {mean_val:.2f} — appears to be PERCENTAGE (0-100)")
    else:
        print(f"  ? {col}: mean = {mean_val:.2f} — UNCLEAR")

# ── 2f. Redundancy check ────────────────────────────────────────────────────
print(f"\n--- Redundancy check ---")
# bullish_avg, bullish_avg_plus_sd, bullish_avg_minus_sd are AAII's own
# historical summaries — not useful as predictive factors
print(f"  Columns that are AAII backward-looking summaries (not predictive factors):")
summary_cols = ['bullish_avg', 'bullish_avg_plus_sd', 'bullish_avg_minus_sd', 'total']
for col in summary_cols:
    if col in df.columns:
        print(f"    - {col}")

print(f"\n  Columns that duplicate data from other sources:")
for col in sp_cols:
    if col in df.columns:
        print(f"    - {col} (redundant with CRSP/FRED S&P 500 data)")

# ── 2g. Duplicate dates ─────────────────────────────────────────────────────
print(f"\n--- Duplicate dates ---")
n_dupes = df['date'].duplicated().sum()
if n_dupes == 0:
    print(f"  ✓ No duplicate dates")
else:
    print(f"  ⚠ {n_dupes} duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED BEFORE CLEANING")
print("=" * 90)

print(f"""
Review the output above and decide:

1. COLUMNS TO DROP:
   - total (just bullish + neutral + bearish = ~100, no information)
   - bullish_avg, bullish_avg_plus_sd, bullish_avg_minus_sd
     (AAII's own historical summaries — not predictive factors,
      and we'd compute our own rolling stats in the model anyway)
   - sp500_weekly_high, sp500_weekly_low, sp500_weekly_close
     (redundant with CRSP and FRED S&P 500 data already in pipeline)

2. COLUMNS TO KEEP (core sentiment factors):
   - bullish, neutral, bearish (raw survey responses)
   - bullish_8w_ma (AAII's 8-week moving average — useful smoothed signal)
   - bull_bear_spread (bullish minus bearish — the headline indicator)

3. DATE RANGE:
   - Trim to 2004-01-01? Already filtered in collection.

4. PUBLICATION LAG:
   - AAII surveys are released on Thursday
   - No available_date column exists
   - The merge pipeline must handle this

5. NaN HANDLING:
   - Depends on what Stage 1 shows

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — AAII Sentiment

Shape: 1,095 rows × 14 columns
Date range: 2004-01-01 → 2024-12-26
Unique dates: 1,095

Columns and dtypes (13 factors):
    1. bullish                        float64        
    2. neutral                        float64        
    3. bearish                        float64        
    4. total                          float64        
    5. bullish_8w_ma                  float64        
    6. bull_bear_spread               float64        
    7. bullish_avg                    float64        
    8. bullish_avg_plus_sd            float64        
    9. bullish_avg_minus_sd           float64        
   10. sp500_weekly_high              float64        
   11. sp500_weekly_low               float64        
   12. sp500_weekly_close             float64        
   13. extra_0                        object         

--- Head (10 rows) ---
      date  bullish  neutral  bearish  total  bullish_8w_ma  bull_bear_spread  bullish_avg  bullish_avg_plus_sd

In [2]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Cleaning decisions and rationale:**
#
# **Columns dropped (8):**
# - `extra_0` — junk overflow column from header parsing. 100% None (object dtype).
# - `total` — always equals `bullish + neutral + bearish` = 1.0000 ± 0.0001.
#   Carries zero information.
# - `bullish_avg`, `bullish_avg_plus_sd`, `bullish_avg_minus_sd` — constants.
#   These are AAII's fixed historical averages printed identically on every row
#   of their spreadsheet (0.376304, 0.476562, 0.276046 respectively). They do
#   not vary over time and have no predictive value.
# - `sp500_weekly_high`, `sp500_weekly_low`, `sp500_weekly_close` — redundant
#   with S&P 500 data already in the pipeline from CRSP daily returns and
#   Fama-French market factor.
#
# **No NaN handling needed:** all 5 retained columns have zero NaN across all
# 1,095 rows.
#
# **Units left as decimal (0–1):** the collection code converted "62.41%" to
# 0.6241. This is internally consistent — bullish + neutral + bearish = 1.0
# for every row. No conversion to percentage applied since the model
# normalises all inputs anyway.
#
# **Date range:** already 2004-01-01 to 2024-12-26 from collection. No trim needed.
#
# **Publication lag not enforced here:** AAII releases on Thursday. The merge
# pipeline must handle this when forward-filling to daily.
#
# **Factors retained: 5** (was 13 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

# ── 4a. Drop columns ────────────────────────────────────────────────────────
drop_cols = [
    'extra_0',                    # junk — 100% None
    'total',                      # always 1.0, no information
    'bullish_avg',                # constant (0.376304 every row)
    'bullish_avg_plus_sd',        # constant (0.476562 every row)
    'bullish_avg_minus_sd',       # constant (0.276046 every row)
    'sp500_weekly_high',          # redundant with CRSP
    'sp500_weekly_low',           # redundant with CRSP
    'sp500_weekly_close',         # redundant with CRSP
]

drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols = [c for c in df.columns if c != 'date']
print(f"\n  Dropped {len(drop_cols_present)} columns: {drop_cols_present}")
print(f"  Remaining: {len(factor_cols)} factor columns")

# ── 4b. Final NaN check ─────────────────────────────────────────────────────
nan_total = df[factor_cols].isna().sum().sum()
if nan_total == 0:
    print(f"\n  ✓ Zero NaN — all clean")
else:
    print(f"\n  ⚠ {nan_total} NaN remaining")

# ── 4c. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    print(f"    {i:>3d}. {c}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))
print(f"\n  Sample (last 5 rows):")
print(df.tail(5).to_string(index=False))

# ── 4d. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'aaii_sentiment_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Dropped 8 columns: ['extra_0', 'total', 'bullish_avg', 'bullish_avg_plus_sd', 'bullish_avg_minus_sd', 'sp500_weekly_high', 'sp500_weekly_low', 'sp500_weekly_close']
  Remaining: 5 factor columns

  ✓ Zero NaN — all clean

  Final shape: 1,095 rows × 6 columns
  Factor columns: 5
  Date range: 2004-01-01 → 2024-12-26

  Factor list (5 columns):
      1. bullish
      2. neutral
      3. bearish
      4. bullish_8w_ma
      5. bull_bear_spread

  Sample (first 5 rows):
      date  bullish  neutral  bearish  bullish_8w_ma  bull_bear_spread
2004-01-01   0.6241   0.2411   0.1348       0.599625            0.4893
2004-01-08   0.6716   0.1493   0.1791       0.616912            0.4925
2004-01-15   0.6629   0.2360   0.1011       0.633237            0.5618
2004-01-22   0.6951   0.1707   0.1341       0.644650            0.5610
2004-01-29   0.5688   0.2813   0.1500       0.629037            0.4188

  Sample (last 5 rows):
      date  bullish  neutral  bearish  bullish_8w_ma